In [1]:
# Import libraries

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [2]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_CLAIMS_DIR = PROJECT_ROOT / "data" / "01_raw" / "cms_claims"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02_preprocessed"
FEATURES_DIR = PROJECT_ROOT / "data" / "03_features"

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_CLAIMS_DIR:", RAW_CLAIMS_DIR)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("FEATURES_DIR:", FEATURES_DIR)

PROJECT_ROOT: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform
RAW_CLAIMS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\01_raw\cms_claims
PREPROCESSED_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed
FEATURES_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\03_features


In [3]:
# Find all files in the raw CMS claims folder

claim_files = sorted(RAW_CLAIMS_DIR.rglob("*"))

print("Files found:", len(claim_files))

for i, file_path in enumerate(claim_files):
    if file_path.is_file():
        print(i, file_path.relative_to(PROJECT_ROOT))

Files found: 6
0 data\01_raw\cms_claims\.gitkeep
1 data\01_raw\cms_claims\DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
2 data\01_raw\cms_claims\DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
3 data\01_raw\cms_claims\DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv
4 data\01_raw\cms_claims\DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
5 data\01_raw\cms_claims\DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv


In [4]:
# Find CSV files, including uppercase CSV extensions

csv_files = sorted(
    [
        path for path in RAW_CLAIMS_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() == ".csv"
    ]
)

print("CSV files found:", len(csv_files))

for i, file_path in enumerate(csv_files):
    print(i, file_path.relative_to(PROJECT_ROOT))

CSV files found: 5
0 data\01_raw\cms_claims\DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
1 data\01_raw\cms_claims\DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
2 data\01_raw\cms_claims\DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv
3 data\01_raw\cms_claims\DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
4 data\01_raw\cms_claims\DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv


In [5]:
# Detect claims files with stricter filename matching

def find_files_by_keywords(files, keywords):
    matches = []

    for file_path in files:
        text = str(file_path).lower()

        if any(keyword.lower() in text for keyword in keywords):
            matches.append(file_path)

    return matches


beneficiary_matches = find_files_by_keywords(
    csv_files,
    [
        "beneficiary_summary",
        "beneficiary summary",
        "bene_summary",
    ],
)

inpatient_matches = find_files_by_keywords(
    csv_files,
    [
        "inpatient_claims",
        "inpatient claims",
        "inpatient",
    ],
)

outpatient_matches = find_files_by_keywords(
    csv_files,
    [
        "outpatient_claims",
        "outpatient claims",
        "outpatient",
    ],
)

print("Beneficiary matches:")
for i, file_path in enumerate(beneficiary_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

print("\nInpatient matches:")
for i, file_path in enumerate(inpatient_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

print("\nOutpatient matches:")
for i, file_path in enumerate(outpatient_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

Beneficiary matches:
0 data\01_raw\cms_claims\DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
1 data\01_raw\cms_claims\DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
2 data\01_raw\cms_claims\DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv

Inpatient matches:
0 data\01_raw\cms_claims\DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv

Outpatient matches:
0 data\01_raw\cms_claims\DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv


In [6]:
# Select the correct files

if len(beneficiary_matches) == 0:
    raise FileNotFoundError("No beneficiary file found.")

if len(inpatient_matches) == 0:
    raise FileNotFoundError("No inpatient claims file found.")

if len(outpatient_matches) == 0:
    raise FileNotFoundError("No outpatient claims file found.")

beneficiary_file = beneficiary_matches[0]
inpatient_file = inpatient_matches[0]
outpatient_file = outpatient_matches[0]

print("Beneficiary file:", beneficiary_file.relative_to(PROJECT_ROOT))
print("Inpatient file:", inpatient_file.relative_to(PROJECT_ROOT))
print("Outpatient file:", outpatient_file.relative_to(PROJECT_ROOT))

Beneficiary file: data\01_raw\cms_claims\DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
Inpatient file: data\01_raw\cms_claims\DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
Outpatient file: data\01_raw\cms_claims\DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv


In [7]:
# Load raw claims datasets

beneficiary_df = None

if beneficiary_file is not None:
    beneficiary_df = pd.read_csv(beneficiary_file, low_memory=False)
    print("Beneficiary shape:", beneficiary_df.shape)
    display(beneficiary_df.head())
else:
    print("Beneficiary file not found. Continuing with inpatient and outpatient claims only.")

inpatient_df = pd.read_csv(inpatient_file, low_memory=False)
outpatient_df = pd.read_csv(outpatient_file, low_memory=False)

print("Inpatient shape:", inpatient_df.shape)
print("Outpatient shape:", outpatient_df.shape)

display(inpatient_df.head())
display(outpatient_df.head())

Beneficiary shape: (116352, 32)


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,BENE_HMO_CVRAGE_TOT_MONS,PLAN_CVRG_MOS_NUM,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,SP_DEPRESSN,SP_DIABETES,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
0,00013D2EFD8E45D1,19230501,NaN,1,1,0,26,950,12,12,12,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
1,00016F745862898F,19430101,NaN,1,1,0,39,230,12,12,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
2,0001FDD721E223DC,19360901,NaN,2,1,0,39,280,12,12,0,12,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,00021CA6FF03E670,19410601,NaN,1,5,0,6,290,0,0,0,0,2,2,2,2,2,2,2,2,2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,00024B3D2352D2D0,19360801,NaN,1,1,0,52,590,12,12,0,0,2,2,2,2,2,2,2,2,1,2,2,0.0,0.0,0.0,30.0,40.0,0.0,220.0,80.0,0.0


Inpatient shape: (66773, 81)
Outpatient shape: (790790, 76)


,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,OT_PHYSN_NPI,CLM_ADMSN_DT,ADMTNG_ICD9_DGNS_CD,CLM_PASS_THRU_PER_DIEM_AMT,NCH_BENE_IP_DDCTBL_AMT,NCH_BENE_PTA_COINSRNC_LBLTY_AM,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,CLM_UTLZTN_DAY_CNT,NCH_BENE_DSCHRG_DT,CLM_DRG_CD,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,ICD9_DGNS_CD_9,ICD9_DGNS_CD_10,ICD9_PRCDR_CD_1,ICD9_PRCDR_CD_2,ICD9_PRCDR_CD_3,ICD9_PRCDR_CD_4,ICD9_PRCDR_CD_5,ICD9_PRCDR_CD_6,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,HCPCS_CD_14,HCPCS_CD_15,HCPCS_CD_16,HCPCS_CD_17,HCPCS_CD_18,HCPCS_CD_19,HCPCS_CD_20,HCPCS_CD_21,HCPCS_CD_22,HCPCS_CD_23,HCPCS_CD_24,HCPCS_CD_25,HCPCS_CD_26,HCPCS_CD_27,HCPCS_CD_28,HCPCS_CD_29,HCPCS_CD_30,HCPCS_CD_31,HCPCS_CD_32,HCPCS_CD_33,HCPCS_CD_34,HCPCS_CD_35,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00013D2EFD8E45D1,196661176988405,1,20100312.0,20100313.0,2600GD,4000.0,0.0,3.139084e+09,NaN,NaN,20100312,4580,0.0,1100.0,0.0,0.0,1.0,20100313,217,7802,78820,V4501,4280,2720,4019,V4502,73300,E9330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00016F745862898F,196201177000368,1,20090412.0,20090418.0,3900MB,26000.0,0.0,6.476809e+09,NaN,NaN,20090412,7866,0.0,1068.0,0.0,0.0,6.0,20090418,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00016F745862898F,196661177015632,1,20090831.0,20090902.0,3900HM,5000.0,0.0,6.119985e+08,6.119985e+08,NaN,20090831,6186,0.0,1068.0,0.0,0.0,2.0,20090902,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,6186,V5866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00016F745862898F,196091176981058,1,20090917.0,20090920.0,3913XU,5000.0,0.0,4.971603e+09,NaN,1.119000e+09,20090917,29590,0.0,1068.0,0.0,0.0,3.0,20090920,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016F745862898F,196261176983265,1,20100626.0,20100701.0,3900MB,16000.0,0.0,6.408400e+09,1.960860e+09,NaN,20100626,5849,0.0,1100.0,0.0,0.0,5.0,20100701,983,3569,4019,3542,V8801,78820,2639,7840,7856,4271,NaN,NaN,E8889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,OT_PHYSN_NPI,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,ICD9_DGNS_CD_9,ICD9_DGNS_CD_10,ICD9_PRCDR_CD_1,ICD9_PRCDR_CD_2,ICD9_PRCDR_CD_3,ICD9_PRCDR_CD_4,ICD9_PRCDR_CD_5,ICD9_PRCDR_CD_6,NCH_BENE_PTB_DDCTBL_AMT,NCH_BENE_PTB_COINSRNC_AMT,ADMTNG_ICD9_DGNS_CD,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,HCPCS_CD_14,HCPCS_CD_15,HCPCS_CD_16,HCPCS_CD_17,HCPCS_CD_18,HCPCS_CD_19,HCPCS_CD_20,HCPCS_CD_21,HCPCS_CD_22,HCPCS_CD_23,HCPCS_CD_24,HCPCS_CD_25,HCPCS_CD_26,HCPCS_CD_27,HCPCS_CD_28,HCPCS_CD_29,HCPCS_CD_30,HCPCS_CD_31,HCPCS_CD_32,HCPCS_CD_33,HCPCS_CD_34,HCPCS_CD_35,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00013D2EFD8E45D1,542192281063886,1,20080904.0,20080904.0,2600RA,50.0,0.0,4.824842e+09,NaN,NaN,0.0,V5841,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,10.0,V5883,85610,84153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00016F745862898F,542272281166593,1,20090602.0,20090602.0,3901GS,30.0,0.0,2.963420e+09,NaN,2.963420e+09,0.0,V5832,V5861,2724,3182,V5869,42731,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,85610,80048,80061,82306,96372,87088,85025,80076,84075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00016F745862898F,542282281644416,1,20090623.0,20090623.0,3939PG,30.0,0.0,5.737808e+09,NaN,5.737808e+09,0.0,9594,E9174,4019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,70.0,NaN,71101,78480,94060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0001FDD721E223DC,542642281250669,1,20091011.0,20091011.0,3902NU,30.0,0.0,1.233848e+09,NaN,NaN,0.0,78943,V5866,V1272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,50.0,56409,36415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00024B3D2352D2D0,542242281386963,1,20080712.0,20080712.0,5200TV,30.0,0.0,9.688809e+09,NaN,NaN,0.0,6009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,40.0,60021,76872,82365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Inspect columns for each dataset

def inspect_columns(df, dataset_name):
    columns_df = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "dtype": [df[col].dtype for col in df.columns],
        "missing_count": [df[col].isna().sum() for col in df.columns],
        "missing_pct": [(df[col].isna().mean() * 100).round(2) for col in df.columns],
    })

    return columns_df.sort_values("missing_pct", ascending=False)


beneficiary_columns = inspect_columns(beneficiary_df, "beneficiary")
inpatient_columns = inspect_columns(inpatient_df, "inpatient")
outpatient_columns = inspect_columns(outpatient_df, "outpatient")

display(beneficiary_columns)
display(inpatient_columns)
display(outpatient_columns)

,dataset,column,dtype,missing_count,missing_pct
2,beneficiary,BENE_DEATH_DT,float64,114538,98.44
0,beneficiary,DESYNPUF_ID,object,0,0.00
1,beneficiary,BENE_BIRTH_DT,int64,0,0.00
3,beneficiary,BENE_SEX_IDENT_CD,int64,0,0.00
4,beneficiary,BENE_RACE_CD,int64,0,0.00
5,beneficiary,BENE_ESRD_IND,object,0,0.00
6,beneficiary,SP_STATE_CODE,int64,0,0.00
7,beneficiary,BENE_COUNTY_CD,int64,0,0.00
8,beneficiary,BENE_HI_CVRAGE_TOT_MONS,int64,0,0.00
9,beneficiary,BENE_SMI_CVRAGE_TOT_MONS,int64,0,0.00


,dataset,column,dtype,missing_count,missing_pct
64,inpatient,HCPCS_CD_29,float64,66773,100.0
65,inpatient,HCPCS_CD_30,float64,66773,100.0
66,inpatient,HCPCS_CD_31,float64,66773,100.0
67,inpatient,HCPCS_CD_32,float64,66773,100.0
36,inpatient,HCPCS_CD_1,float64,66773,100.0
...,...,...,...,...,...
11,inpatient,CLM_ADMSN_DT,int64,0,0.0
1,inpatient,CLM_ID,int64,0,0.0
5,inpatient,PRVDR_NUM,object,0,0.0
18,inpatient,NCH_BENE_DSCHRG_DT,int64,0,0.0


,dataset,column,dtype,missing_count,missing_pct
26,outpatient,ICD9_PRCDR_CD_5,object,790755,100.00
75,outpatient,HCPCS_CD_45,float64,790790,100.00
27,outpatient,ICD9_PRCDR_CD_6,object,790761,100.00
25,outpatient,ICD9_PRCDR_CD_4,object,790742,99.99
24,outpatient,ICD9_PRCDR_CD_3,object,790717,99.99
...,...,...,...,...,...
7,outpatient,NCH_PRMRY_PYR_CLM_PD_AMT,float64,0,0.00
1,outpatient,CLM_ID,int64,0,0.00
0,outpatient,DESYNPUF_ID,object,0,0.00
28,outpatient,NCH_BENE_PTB_DDCTBL_AMT,float64,0,0.00


In [9]:
# Search columns by keyword

def search_columns(df, keywords):
    matches = []

    for col in df.columns:
        col_lower = col.lower()

        if any(keyword.lower() in col_lower for keyword in keywords):
            matches.append(col)

    return matches


keywords = {
    "member_id": ["desynpuf_id", "bene", "beneficiary"],
    "claim_id": ["claim", "clm"],
    "date": ["date", "dt"],
    "payment": ["payment", "pmt", "paid", "reimb"],
    "diagnosis": ["diag", "icd"],
    "provider": ["provider", "prvdr"],
}

for label, terms in keywords.items():
    print(f"\nBENEFICIARY - {label}")
    print(search_columns(beneficiary_df, terms))

    print(f"INPATIENT - {label}")
    print(search_columns(inpatient_df, terms))

    print(f"OUTPATIENT - {label}")
    print(search_columns(outpatient_df, terms))


BENEFICIARY - member_id
['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS']
INPATIENT - member_id
['DESYNPUF_ID', 'NCH_BENE_IP_DDCTBL_AMT', 'NCH_BENE_PTA_COINSRNC_LBLTY_AM', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'NCH_BENE_DSCHRG_DT']
OUTPATIENT - member_id
['DESYNPUF_ID', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT']

BENEFICIARY - claim_id
[]
INPATIENT - claim_id
['CLM_ID', 'CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'CLM_ADMSN_DT', 'CLM_PASS_THRU_PER_DIEM_AMT', 'CLM_UTLZTN_DAY_CNT', 'CLM_DRG_CD']
OUTPATIENT - claim_id
['CLM_ID', 'CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT']

BENEFICIARY - date
['BENE_BIRTH_DT', 'BENE_DEATH_DT']
INPATIENT - date
['CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_ADMSN_DT', 'NCH_BENE_DSCHRG_DT']
OUTPATIENT - date
['CLM_FR

In [10]:
# Helper function to safely select columns

def get_first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col

    return None

In [11]:
# Map inpatient columns to clean names

inpatient_column_map = {
    "member_id": get_first_existing_column(
        inpatient_df,
        ["DESYNPUF_ID", "BENE_ID"],
    ),
    "claim_id": get_first_existing_column(
        inpatient_df,
        ["CLM_ID"],
    ),
    "claim_start_date": get_first_existing_column(
        inpatient_df,
        ["CLM_FROM_DT", "CLM_THRU_DT"],
    ),
    "claim_end_date": get_first_existing_column(
        inpatient_df,
        ["CLM_THRU_DT"],
    ),
    "claim_payment_amount": get_first_existing_column(
        inpatient_df,
        ["CLM_PMT_AMT", "CLM_TOT_CHRG_AMT", "NCH_PRMRY_PYR_CLM_PD_AMT"],
    ),
    "provider_id": get_first_existing_column(
        inpatient_df,
        ["PRVDR_NUM"],
    ),
    "primary_diagnosis_code": get_first_existing_column(
        inpatient_df,
        ["ADMTNG_ICD9_DGNS_CD", "ICD9_DGNS_CD_1"],
    ),
}

inpatient_column_map

{'member_id': 'DESYNPUF_ID',
 'claim_id': 'CLM_ID',
 'claim_start_date': 'CLM_FROM_DT',
 'claim_end_date': 'CLM_THRU_DT',
 'claim_payment_amount': 'CLM_PMT_AMT',
 'provider_id': 'PRVDR_NUM',
 'primary_diagnosis_code': 'ADMTNG_ICD9_DGNS_CD'}

In [12]:
# Map outpatient columns to clean names

outpatient_column_map = {
    "member_id": get_first_existing_column(
        outpatient_df,
        ["DESYNPUF_ID", "BENE_ID"],
    ),
    "claim_id": get_first_existing_column(
        outpatient_df,
        ["CLM_ID"],
    ),
    "claim_start_date": get_first_existing_column(
        outpatient_df,
        ["CLM_FROM_DT", "CLM_THRU_DT"],
    ),
    "claim_end_date": get_first_existing_column(
        outpatient_df,
        ["CLM_THRU_DT"],
    ),
    "claim_payment_amount": get_first_existing_column(
        outpatient_df,
        ["CLM_PMT_AMT", "CLM_TOT_CHRG_AMT", "NCH_PRMRY_PYR_CLM_PD_AMT"],
    ),
    "provider_id": get_first_existing_column(
        outpatient_df,
        ["PRVDR_NUM"],
    ),
    "primary_diagnosis_code": get_first_existing_column(
        outpatient_df,
        ["ICD9_DGNS_CD_1"],
    ),
}

outpatient_column_map

{'member_id': 'DESYNPUF_ID',
 'claim_id': 'CLM_ID',
 'claim_start_date': 'CLM_FROM_DT',
 'claim_end_date': 'CLM_THRU_DT',
 'claim_payment_amount': 'CLM_PMT_AMT',
 'provider_id': 'PRVDR_NUM',
 'primary_diagnosis_code': 'ICD9_DGNS_CD_1'}

In [13]:
# Build clean inpatient claims dataset

clean_inpatient_df = pd.DataFrame()

for clean_col, source_col in inpatient_column_map.items():
    if source_col is None:
        clean_inpatient_df[clean_col] = np.nan
    else:
        clean_inpatient_df[clean_col] = inpatient_df[source_col]

clean_inpatient_df["claim_type"] = "inpatient"

print("Clean inpatient shape:", clean_inpatient_df.shape)
display(clean_inpatient_df.head())

Clean inpatient shape: (66773, 8)


,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type
0,00013D2EFD8E45D1,196661176988405,20100312.0,20100313.0,4000.0,2600GD,4580,inpatient
1,00016F745862898F,196201177000368,20090412.0,20090418.0,26000.0,3900MB,7866,inpatient
2,00016F745862898F,196661177015632,20090831.0,20090902.0,5000.0,3900HM,6186,inpatient
3,00016F745862898F,196091176981058,20090917.0,20090920.0,5000.0,3913XU,29590,inpatient
4,00016F745862898F,196261176983265,20100626.0,20100701.0,16000.0,3900MB,5849,inpatient


In [14]:
# Build clean outpatient claims dataset

clean_outpatient_df = pd.DataFrame()

for clean_col, source_col in outpatient_column_map.items():
    if source_col is None:
        clean_outpatient_df[clean_col] = np.nan
    else:
        clean_outpatient_df[clean_col] = outpatient_df[source_col]

clean_outpatient_df["claim_type"] = "outpatient"

print("Clean outpatient shape:", clean_outpatient_df.shape)
display(clean_outpatient_df.head())

Clean outpatient shape: (790790, 8)


,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type
0,00013D2EFD8E45D1,542192281063886,20080904.0,20080904.0,50.0,2600RA,V5841,outpatient
1,00016F745862898F,542272281166593,20090602.0,20090602.0,30.0,3901GS,V5832,outpatient
2,00016F745862898F,542282281644416,20090623.0,20090623.0,30.0,3939PG,9594,outpatient
3,0001FDD721E223DC,542642281250669,20091011.0,20091011.0,30.0,3902NU,78943,outpatient
4,00024B3D2352D2D0,542242281386963,20080712.0,20080712.0,30.0,5200TV,6009,outpatient


In [15]:
# Combine inpatient and outpatient claims

claims_df = pd.concat(
    [
        clean_inpatient_df,
        clean_outpatient_df,
    ],
    ignore_index=True,
)

print("Combined claims shape:", claims_df.shape)
display(claims_df.head())

Combined claims shape: (857563, 8)


,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type
0,00013D2EFD8E45D1,196661176988405,20100312.0,20100313.0,4000.0,2600GD,4580,inpatient
1,00016F745862898F,196201177000368,20090412.0,20090418.0,26000.0,3900MB,7866,inpatient
2,00016F745862898F,196661177015632,20090831.0,20090902.0,5000.0,3900HM,6186,inpatient
3,00016F745862898F,196091176981058,20090917.0,20090920.0,5000.0,3913XU,29590,inpatient
4,00016F745862898F,196261176983265,20100626.0,20100701.0,16000.0,3900MB,5849,inpatient


In [16]:
# Clean text fields

text_cols = [
    "member_id",
    "claim_id",
    "provider_id",
    "primary_diagnosis_code",
    "claim_type",
]

for col in text_cols:
    claims_df[col] = (
        claims_df[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
        })
    )

display(claims_df.head())

,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type
0,00013D2EFD8E45D1,196661176988405,20100312.0,20100313.0,4000.0,2600GD,4580,inpatient
1,00016F745862898F,196201177000368,20090412.0,20090418.0,26000.0,3900MB,7866,inpatient
2,00016F745862898F,196661177015632,20090831.0,20090902.0,5000.0,3900HM,6186,inpatient
3,00016F745862898F,196091176981058,20090917.0,20090920.0,5000.0,3913XU,29590,inpatient
4,00016F745862898F,196261176983265,20100626.0,20100701.0,16000.0,3900MB,5849,inpatient


In [17]:
# Clean date fields correctly
# CMS DE-SynPUF dates are formatted like YYYYMMDD.

date_cols = [
    "claim_start_date",
    "claim_end_date",
]

for col in date_cols:
    claims_df[col] = (
        claims_df[col]
        .astype("string")
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    claims_df[col] = pd.to_datetime(
        claims_df[col],
        format="%Y%m%d",
        errors="coerce",
    )

claims_df["claim_duration_days"] = (
    claims_df["claim_end_date"] - claims_df["claim_start_date"]
).dt.days + 1

claims_df["claim_duration_days"] = claims_df["claim_duration_days"].clip(lower=1)

display(
    claims_df[
        [
            "claim_start_date",
            "claim_end_date",
            "claim_duration_days",
        ]
    ].head()
)

,claim_start_date,claim_end_date,claim_duration_days
0,2010-03-12,2010-03-13,2.0
1,2009-04-12,2009-04-18,7.0
2,2009-08-31,2009-09-02,3.0
3,2009-09-17,2009-09-20,4.0
4,2010-06-26,2010-07-01,6.0


In [18]:
# Clean payment amount

def clean_amount(value):
    if pd.isna(value):
        return np.nan

    value = str(value).replace("$", "").replace(",", "").strip()

    try:
        return float(value)
    except ValueError:
        return np.nan


claims_df["claim_payment_amount"] = claims_df["claim_payment_amount"].apply(clean_amount)

display(claims_df["claim_payment_amount"].describe())

count    857563.000000
mean       1007.255315
std        3640.546992
min       -8000.000000
25%          40.000000
50%          80.000000
75%         300.000000
max       57000.000000
Name: claim_payment_amount, dtype: float64

In [19]:
# Check missing values after cleaning

missing_summary = claims_df.isna().sum().reset_index()
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(claims_df) * 100).round(2)

display(missing_summary.sort_values("missing_pct", ascending=False))

,column,missing_count,missing_pct
2,claim_start_date,11321,1.32
8,claim_duration_days,11321,1.32
3,claim_end_date,11321,1.32
6,primary_diagnosis_code,6219,0.73
0,member_id,0,0.00
4,claim_payment_amount,0,0.00
1,claim_id,0,0.00
5,provider_id,0,0.00
7,claim_type,0,0.00


In [20]:
# Remove rows missing core claim fields

before_rows = len(claims_df)

required_cols = [
    "member_id",
    "claim_id",
    "claim_payment_amount",
    "claim_type",
]

claims_df = claims_df.dropna(subset=required_cols).copy()

after_rows = len(claims_df)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

display(claims_df.head())

Rows before: 857563
Rows after: 857563
Rows removed: 0


,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type,claim_duration_days
0,00013D2EFD8E45D1,196661176988405,2010-03-12,2010-03-13,4000.0,2600GD,4580,inpatient,2.0
1,00016F745862898F,196201177000368,2009-04-12,2009-04-18,26000.0,3900MB,7866,inpatient,7.0
2,00016F745862898F,196661177015632,2009-08-31,2009-09-02,5000.0,3900HM,6186,inpatient,3.0
3,00016F745862898F,196091176981058,2009-09-17,2009-09-20,5000.0,3913XU,29590,inpatient,4.0
4,00016F745862898F,196261176983265,2010-06-26,2010-07-01,16000.0,3900MB,5849,inpatient,6.0


In [21]:
# Basic claims summary

print("Total claims:", len(claims_df))
print("Unique members:", claims_df["member_id"].nunique())
print("Unique providers:", claims_df["provider_id"].nunique())
print("Claim types:", claims_df["claim_type"].value_counts().to_dict())

display(claims_df["claim_payment_amount"].describe())
display(claims_df["claim_type"].value_counts().to_frame("count"))

Total claims: 857563
Unique members: 86738
Unique providers: 6801
Claim types: {'outpatient': 790790, 'inpatient': 66773}


count    857563.000000
mean       1007.255315
std        3640.546992
min       -8000.000000
25%          40.000000
50%          80.000000
75%         300.000000
max       57000.000000
Name: claim_payment_amount, dtype: float64

,count
claim_type,
outpatient,790790
inpatient,66773


In [22]:
# Create high-cost claim target
# This will be used in Notebook 3 for modeling.

high_cost_threshold = claims_df["claim_payment_amount"].quantile(0.90)

claims_df["high_cost_claim"] = (
    claims_df["claim_payment_amount"] >= high_cost_threshold
).astype(int)

print("High-cost threshold:", high_cost_threshold)
display(claims_df["high_cost_claim"].value_counts(normalize=True).to_frame("rate"))
display(claims_df[["claim_payment_amount", "high_cost_claim"]].head())

High-cost threshold: 2100.0


,rate
high_cost_claim,
0,0.89759
1,0.10241


,claim_payment_amount,high_cost_claim
0,4000.0,1
1,26000.0,1
2,5000.0,1
3,5000.0,1
4,16000.0,1


In [23]:
# Create member-level utilization features

member_features_df = (
    claims_df
    .groupby("member_id")
    .agg(
        total_claims=("claim_id", "nunique"),
        total_claim_payment=("claim_payment_amount", "sum"),
        avg_claim_payment=("claim_payment_amount", "mean"),
        max_claim_payment=("claim_payment_amount", "max"),
        inpatient_claims=("claim_type", lambda x: (x == "inpatient").sum()),
        outpatient_claims=("claim_type", lambda x: (x == "outpatient").sum()),
        unique_providers=("provider_id", "nunique"),
        unique_diagnoses=("primary_diagnosis_code", "nunique"),
        avg_claim_duration_days=("claim_duration_days", "mean"),
        high_cost_claims=("high_cost_claim", "sum"),
    )
    .reset_index()
)

member_features_df["has_high_cost_claim"] = (
    member_features_df["high_cost_claims"] > 0
).astype(int)

display(member_features_df.head())

,member_id,total_claims,total_claim_payment,avg_claim_payment,max_claim_payment,inpatient_claims,outpatient_claims,unique_providers,unique_diagnoses,avg_claim_duration_days,high_cost_claims,has_high_cost_claim
0,00013D2EFD8E45D1,2,4050.0,2025.000000,4000.0,1,1,2,2,1.500000,1,1
1,00016F745862898F,6,52060.0,8676.666667,26000.0,4,2,5,6,3.666667,4,1
2,0001FDD721E223DC,1,30.0,30.000000,30.0,0,1,1,1,1.000000,0,0
3,00024B3D2352D2D0,4,160.0,40.000000,80.0,0,4,4,4,1.000000,0,0
4,0002F28CE057345B,18,2920.0,153.684211,700.0,0,19,4,18,3.277778,0,0


In [24]:
# Create claim-level modeling features
# Negative payments are clipped to 0 for stable log transformation.

claim_features_df = claims_df.copy()

claim_features_df["claim_payment_amount_clipped"] = claim_features_df["claim_payment_amount"].clip(lower=0)

claim_features_df["claim_payment_log"] = np.log1p(
    claim_features_df["claim_payment_amount_clipped"]
)

claim_features_df["has_provider_id"] = claim_features_df["provider_id"].notna().astype(int)
claim_features_df["has_diagnosis_code"] = claim_features_df["primary_diagnosis_code"].notna().astype(int)

display(claim_features_df.head())

,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type,claim_duration_days,high_cost_claim,claim_payment_amount_clipped,claim_payment_log,has_provider_id,has_diagnosis_code
0,00013D2EFD8E45D1,196661176988405,2010-03-12,2010-03-13,4000.0,2600GD,4580,inpatient,2.0,1,4000.0,8.294300,1,1
1,00016F745862898F,196201177000368,2009-04-12,2009-04-18,26000.0,3900MB,7866,inpatient,7.0,1,26000.0,10.165890,1,1
2,00016F745862898F,196661177015632,2009-08-31,2009-09-02,5000.0,3900HM,6186,inpatient,3.0,1,5000.0,8.517393,1,1
3,00016F745862898F,196091176981058,2009-09-17,2009-09-20,5000.0,3913XU,29590,inpatient,4.0,1,5000.0,8.517393,1,1
4,00016F745862898F,196261176983265,2010-06-26,2010-07-01,16000.0,3900MB,5849,inpatient,6.0,1,16000.0,9.680406,1,1


In [25]:
# Save cleaned claims and features

clean_claims_output_path = PREPROCESSED_DIR / "clean_claims_data.csv"
claim_features_output_path = FEATURES_DIR / "claim_features.csv"
member_features_output_path = FEATURES_DIR / "member_claim_features.csv"

claims_df.to_csv(clean_claims_output_path, index=False)
claim_features_df.to_csv(claim_features_output_path, index=False)
member_features_df.to_csv(member_features_output_path, index=False)

print("Saved:", clean_claims_output_path)
print("Saved:", claim_features_output_path)
print("Saved:", member_features_output_path)

Saved: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\clean_claims_data.csv
Saved: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\03_features\claim_features.csv
Saved: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\03_features\member_claim_features.csv


In [26]:
# Reload saved files to confirm they work

check_claims_df = pd.read_csv(clean_claims_output_path)
check_claim_features_df = pd.read_csv(claim_features_output_path)
check_member_features_df = pd.read_csv(member_features_output_path)

print("Clean claims shape:", check_claims_df.shape)
print("Claim features shape:", check_claim_features_df.shape)
print("Member features shape:", check_member_features_df.shape)

display(check_claim_features_df.head())

Clean claims shape: (857563, 10)
Claim features shape: (857563, 14)
Member features shape: (86738, 12)


,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type,claim_duration_days,high_cost_claim,claim_payment_amount_clipped,claim_payment_log,has_provider_id,has_diagnosis_code
0,00013D2EFD8E45D1,196661176988405,2010-03-12,2010-03-13,4000.0,2600GD,4580,inpatient,2.0,1,4000.0,8.294300,1,1
1,00016F745862898F,196201177000368,2009-04-12,2009-04-18,26000.0,3900MB,7866,inpatient,7.0,1,26000.0,10.165890,1,1
2,00016F745862898F,196661177015632,2009-08-31,2009-09-02,5000.0,3900HM,6186,inpatient,3.0,1,5000.0,8.517393,1,1
3,00016F745862898F,196091176981058,2009-09-17,2009-09-20,5000.0,3913XU,29590,inpatient,4.0,1,5000.0,8.517393,1,1
4,00016F745862898F,196261176983265,2010-06-26,2010-07-01,16000.0,3900MB,5849,inpatient,6.0,1,16000.0,9.680406,1,1


In [27]:
# Final notebook summary

print("Notebook 2 complete.")
print("Clean claim records:", len(claims_df))
print("Unique members:", claims_df["member_id"].nunique())
print("High-cost threshold:", high_cost_threshold)

print("\nFiles created:")
print("1. data/02_preprocessed/clean_claims_data.csv")
print("2. data/03_features/claim_features.csv")
print("3. data/03_features/member_claim_features.csv")

Notebook 2 complete.
Clean claim records: 857563
Unique members: 86738
High-cost threshold: 2100.0

Files created:
1. data/02_preprocessed/clean_claims_data.csv
2. data/03_features/claim_features.csv
3. data/03_features/member_claim_features.csv
